# Crew People Ops: KPI layer in SQL

**What this notebook does.** The Crew dashboard (`dashboard_template.html`) is a visual concept generated with Muse. Its KPI numbers were typed into the page, so none of them could be traced to data. This notebook builds the missing layer:

1. a relational schema for a **fictional** 42-person company (synthetic data, seeded, reproducible),
2. SQL that computes every headline KPI, including window functions,
3. data-quality checks, with a negative test that proves the checks catch bad data,
4. three analyst-style insights,
5. the final `index.html`, where every number comes from the SQL below.

**Authorship note.** Layout and 3D scene come from the Muse template. Schema, synthetic data, SQL, KPI definitions, checks and wiring were written AI-assisted and reviewed by the author. All people, candidates and amounts are fictional; nothing here is a real result.

Reference date ("today" on the dashboard): **2026-10-31**.

In [1]:
import sqlite3, json, re, random, datetime as dt, pathlib
import pandas as pd
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)

REF = dt.date(2026, 10, 31)          # the dashboard's "today"
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
TEMPLATE = pathlib.Path("dashboard_template.html")
DB = sqlite3.connect(":memory:")
def q(sql, **params):
    return pd.read_sql_query(sql, DB, params=params)
def d(s): return dt.date.fromisoformat(s)
print("SQLite", sqlite3.sqlite_version, "| window functions need >= 3.25")

SQLite 3.53.1 | window functions need >= 3.25


## 1. Schema
Seven tables. Primary and foreign keys are enforced; `calendar` supports business-day logic.

In [2]:
DB.executescript('''
PRAGMA foreign_keys = ON;
CREATE TABLE departments (dept_id TEXT PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE employees (
  employee_id INTEGER PRIMARY KEY, name TEXT NOT NULL, role TEXT NOT NULL,
  dept_id TEXT NOT NULL REFERENCES departments(dept_id), level INTEGER NOT NULL CHECK(level BETWEEN 1 AND 5),
  location TEXT, manager_id INTEGER REFERENCES employees(employee_id),
  hire_date TEXT NOT NULL, term_date TEXT, annual_base_salary INTEGER NOT NULL,
  pto_total_days INTEGER, pto_left_days INTEGER);
CREATE TABLE time_off (employee_id INTEGER NOT NULL REFERENCES employees(employee_id),
  leave_date TEXT NOT NULL, leave_type TEXT NOT NULL CHECK(leave_type IN ('vac','sick','par','per')),
  PRIMARY KEY (employee_id, leave_date));
CREATE TABLE requisitions (req_id TEXT PRIMARY KEY, title TEXT NOT NULL,
  dept_id TEXT NOT NULL REFERENCES departments(dept_id), opened_date TEXT NOT NULL, closed_date TEXT,
  status TEXT NOT NULL CHECK(status IN ('open','filled','closed_no_hire')));
CREATE TABLE candidates (candidate_id INTEGER PRIMARY KEY, name TEXT NOT NULL,
  req_id TEXT NOT NULL REFERENCES requisitions(req_id),
  applied_date TEXT NOT NULL, screen_date TEXT, interview_date TEXT, offer_date TEXT, accepted_date TEXT, start_date TEXT,
  stage TEXT NOT NULL CHECK(stage IN ('Applied','Screen','Interview','Offer','Hired','Rejected','Declined')),
  final_round INTEGER NOT NULL DEFAULT 0);
CREATE TABLE payroll_runs (period_month TEXT NOT NULL, employee_id INTEGER NOT NULL REFERENCES employees(employee_id),
  base_pay REAL NOT NULL, benefits_equity REAL NOT NULL, employer_tax REAL NOT NULL, pay_date TEXT NOT NULL,
  PRIMARY KEY (period_month, employee_id));
CREATE TABLE calendar (day TEXT PRIMARY KEY, is_weekday INTEGER NOT NULL, bd_index INTEGER);
''')
DEPT_NAMES = {"eng":"Engineering","prod":"Product","des":"Design","sales":"Sales","ops":"People & Ops","fin":"Finance"}
DB.executemany("INSERT INTO departments VALUES (?,?)", DEPT_NAMES.items())

# business-day calendar for 2026
day, bd, rows = dt.date(2026,1,1), 0, []
while day <= dt.date(2026,12,31):
    wk = int(day.weekday() < 5); bd += wk
    rows.append((day.isoformat(), wk, bd if wk else None)); day += dt.timedelta(days=1)
DB.executemany("INSERT INTO calendar VALUES (?,?,?)", rows)
print("tables:", [r[0] for r in DB.execute("SELECT name FROM sqlite_master WHERE type='table'")])

tables: ['departments', 'employees', 'time_off', 'requisitions', 'candidates', 'payroll_runs', 'calendar']


## 2. Synthetic employees, leave and salaries
Names, roles, levels, locations and reporting lines come from the Muse template (so the 3D scene still matches). **Hire dates are derived from the template's tenure values**, except two overrides so the dashboard has recent joiners. Salaries, benefits and the six departed employees are added here as synthetic assumptions.

In [3]:
html = TEMPLATE.read_text(encoding="utf8")
RAW = json.loads("[" + re.search(r"var RAW = \[(.*?)\n\s*\];", html, re.S).group(1).strip().rstrip(",") + "]")
assert len(RAW) == 42

HIRE_OVERRIDE = {"Ben Osei": "2026-10-05", "Omar Haddad": "2026-08-17"}   # recent joiners (synthetic)
LEVEL_BASE = {1: 55000, 2: 95000, 3: 125000, 4: 160000, 5: 215000}        # synthetic annual base by level
DEPT_FACTOR = {"eng":1.05, "prod":1.00, "des":0.95, "sales":0.95, "ops":0.85, "fin":0.95}
rng = random.Random(2026)
def salary(dept, level):
    return int(round(LEVEL_BASE[level] * DEPT_FACTOR[dept] * rng.uniform(0.95, 1.05) / 500) * 500)

emp = []
for i, (name, role, dept, level, loc, tenure, mgr_idx, pto_left, pto_total) in enumerate(RAW):
    hire = HIRE_OVERRIDE.get(name) or (REF - dt.timedelta(days=round(tenure * 365.25))).isoformat()
    emp.append((i+1, name, role, dept, level, loc, (mgr_idx+1) if mgr_idx >= 0 else None, hire, None,
                salary(dept, level), pto_total, pto_left))
by_name = {e[1]: e[0] for e in emp}

# six former employees (synthetic): three left in the last 12 months, three in the 12 months before
EX = [("Rene Dubois","Software Engineer","eng",2,"Austin","Priya Nair","2023-02-13","2026-03-20"),
      ("Kyle Brennan","Account Executive","sales",3,"Chicago","Chris Novak","2024-04-08","2026-06-12"),
      ("Tessa Moreau","Product Designer","des",3,"New York","Lena Fischer","2022-09-19","2026-08-28"),
      ("Wes Thornton","Support Specialist","ops",2,"Denver","Julia Brandt","2023-05-01","2025-03-14"),
      ("Mira Patel","Financial Analyst","fin",3,"Seattle","Sara Lindholm","2022-01-10","2025-07-25"),
      ("Ahmed Rahimi","Software Engineer","eng",2,"Remote · PL","Diego Fuentes","2022-02-07","2025-09-12")]
for k, (name, role, dept, level, loc, mgr, hire, term) in enumerate(EX):
    emp.append((43+k, name, role, dept, level, loc, by_name[mgr], hire, term, salary(dept, level), None, None))

DB.executemany("INSERT INTO employees VALUES (?,?,?,?,?,?,?,?,?,?,?,?)", sorted(emp, key=lambda r: (r[7] if False else r[0])))

# leave calendar: parsed from the template's LEAVE object (weekday rows only)
leave_src = re.search(r"var LEAVE = \{(.*?)\n\s*\};", html, re.S).group(1)
TYPE = {"vac":"vac","sick":"sick","par":"par","per":"per"}
leave_rows = []
for day_s, body in re.findall(r'"(\d{4}-\d\d-\d\d)":\[(.*?)\]', leave_src):
    for n, dd, t in re.findall(r'n:"(.*?)",d:"(.*?)",t:"(.*?)"', body):
        leave_rows.append((by_name[n], day_s, t))
DB.executemany("INSERT INTO time_off VALUES (?,?,?)", leave_rows)
print(len(emp), "employees (42 active + 6 former),", len(leave_rows), "leave-days")
q("SELECT dept_id, COUNT(*) active, ROUND(AVG(annual_base_salary)) avg_base FROM employees WHERE term_date IS NULL GROUP BY dept_id")

48 employees (42 active + 6 former), 42 leave-days


,dept_id,active,avg_base
0,des,6,135417.0
1,eng,13,135192.0
2,fin,4,140000.0
3,ops,7,113286.0
4,prod,6,140500.0
5,sales,6,135083.0


## 3. Synthetic hiring pipeline
Nine open requisitions plus closed ones. The named candidates are the ones already shown on the dashboard cards; the rest are anonymous. Stage dates are generated with seeded random gaps, scaled so no date falls after the reference date or before the requisition opened.

In [4]:
REQS = [  # req_id, title, dept, opened, closed, status
 ("R101","Senior Data Analyst","prod","2026-09-08",None,"open"), ("R102","Backend Engineer","eng","2026-08-24",None,"open"),
 ("R103","Backend Engineer","eng","2026-09-14",None,"open"),     ("R104","Product Designer","des","2026-09-21",None,"open"),
 ("R105","Account Executive","sales","2026-08-31",None,"open"),  ("R106","Engineering Manager","eng","2026-07-27",None,"open"),
 ("R107","Financial Analyst","fin","2026-09-28",None,"open"),     ("R108","Customer Success Manager","sales","2026-10-05",None,"open"),
 ("R109","Product Manager","prod","2026-10-12",None,"open"),
 ("R098","People Ops Coordinator","ops","2026-09-01","2026-10-16","filled"), ("R099","QA Engineer","eng","2026-09-07","2026-10-23","filled"),
 ("R095","Software Engineer, Intern","eng","2026-06-15","2026-07-31","filled"), ("R096","Software Engineer","eng","2026-05-04","2026-06-29","filled"),
 ("R091","Senior Product Manager","prod","2026-02-09","2026-04-20","closed_no_hire"), ("R092","Product Designer","des","2026-01-12","2026-03-10","closed_no_hire"),
 ("R093","Account Executive","sales","2026-03-02","2026-05-04","closed_no_hire"), ("R094","Support Specialist","ops","2026-04-13","2026-06-02","closed_no_hire")]
DB.executemany("INSERT INTO requisitions VALUES (?,?,?,?,?,?)", REQS)
OPEN_REQS = [r[0] for r in REQS if r[5] == "open"]
OPENED = {r[0]: d(r[3]) for r in REQS}

crng = random.Random(42)
cands = []   # dicts
def add(name, req, stage, applied, screen=None, interview=None, offer=None, accepted=None, start=None, final=0):
    cands.append(dict(name=name, req_id=req, applied_date=applied, screen_date=screen, interview_date=interview,
                      offer_date=offer, accepted_date=accepted, start_date=start, stage=stage, final_round=final))

# (a) hand-set history: accepted hires (two already started, two start in November) and declined offers
add("Omar Haddad","R096","Hired","2026-05-11","2026-05-18","2026-05-29","2026-06-22","2026-06-29","2026-08-17")
add("Ben Osei","R095","Hired","2026-06-22","2026-06-26","2026-07-08","2026-07-24","2026-07-31","2026-10-05")
add("Kate Nguyen","R098","Hired","2026-09-08","2026-09-14","2026-09-24","2026-10-12","2026-10-16","2026-11-16")
add("Luis Herrera","R099","Hired","2026-09-11","2026-09-17","2026-09-28","2026-10-19","2026-10-23","2026-11-30")
add("Declined Offer 1","R092","Declined","2026-01-20","2026-01-27","2026-02-09","2026-03-02")
add("Declined Offer 2","R093","Declined","2026-03-11","2026-03-17","2026-03-30","2026-04-21")
add("Declined Offer 3","R091","Declined","2026-02-16","2026-02-24","2026-03-10","2026-04-06")
add("Declined Offer 4","R094","Declined","2026-04-20","2026-04-24","2026-05-05","2026-05-21")

# (b) current pipeline: seeded gaps, scaled to fit between requisition open date and the reference date
GAPS = [(3,10), (5,12), (6,20)]            # applied->screen, screen->interview, interview->offer
def current(name, req, stage, final=0):
    reach = ["Applied","Screen","Interview","Offer"].index(stage)
    gaps = [crng.randint(*GAPS[i]) for i in range(reach)]
    dwell = crng.randint(0, 9 if reach else 14)
    age = sum(gaps) + dwell
    cap = (REF - OPENED[req]).days - 1
    if age > cap:
        scale = cap / age; gaps = [max(1, int(g*scale)) for g in gaps]; dwell = int(dwell*scale); age = sum(gaps) + dwell
    t0 = REF - dt.timedelta(days=age); ds = [t0]
    for g in gaps: ds.append(ds[-1] + dt.timedelta(days=g))
    f = [x.isoformat() for x in ds] + [None]*(4-len(ds))
    add(name, req, stage, f[0], f[1], f[2], f[3], final=final)

named = [("Alina Brooks","R101","Screen",0),("James Park","R102","Screen",0),
         ("Chloe Dubois","R104","Interview",1),("Andre Silva","R105","Interview",1),("Mina Choi","R103","Interview",0),
         ("Samira El-Amin","R106","Offer",0),("Jonah Fields","R107","Offer",0),("Priti Menon","R108","Offer",0)]
for n, r, s, f in named: current(n, r, s, f)
anon = [("Applied",18,[0]*18), ("Screen",7,[0]*7), ("Interview",3,[1,1,0])]
k = 0
for stage, n, finals in anon:
    for j in range(n):
        k += 1; current(f"Applicant A-{k:03d}", OPEN_REQS[(k-1) % len(OPEN_REQS)], stage, finals[j])

# (c) rejected candidates (two per open requisition) to give the funnel some realism
for req in OPEN_REQS:
    for j in range(2):
        ap = OPENED[req] + dt.timedelta(days=crng.randint(2, 14))
        sc = ap + dt.timedelta(days=crng.randint(3, 10)); iv = sc + dt.timedelta(days=crng.randint(5, 12))
        reach_iv = crng.random() < 0.5
        if sc > REF: sc = None
        if iv > REF or not reach_iv or sc is None: iv = None
        if ap > REF: continue
        add(f"Rejected {req}-{j+1}", req, "Rejected", ap.isoformat(), sc.isoformat() if sc else None, iv.isoformat() if iv else None)

DB.executemany("INSERT INTO candidates (name, req_id, applied_date, screen_date, interview_date, offer_date, accepted_date, start_date, stage, final_round) "
               "VALUES (:name,:req_id,:applied_date,:screen_date,:interview_date,:offer_date,:accepted_date,:start_date,:stage,:final_round)", cands)
q("SELECT stage, COUNT(*) n FROM candidates GROUP BY stage ORDER BY n DESC")

,stage,n
0,Rejected,18
1,Applied,18
2,Screen,9
3,Interview,6
4,Hired,4
5,Declined,4
6,Offer,3


## 4. Synthetic payroll runs (Apr to Oct 2026)
One row per employee per month, prorated by days employed in the month. Pay date is the first Friday of the following month, so the **October period is paid on Fri Nov 6** (the dashboard's "next run").

Simplifying assumptions (stated, not real tax or benefits rules): benefits and equity = 8/15/20/25/30% of base by level 1 to 5; employer taxes = flat 7% of base.

In [5]:
BEN_PCT = {1:0.08, 2:0.15, 3:0.20, 4:0.25, 5:0.30}; TAX = 0.07
def first_friday(y, m):
    x = dt.date(y, m, 1)
    while x.weekday() != 4: x += dt.timedelta(days=1)
    return x
pay_rows = []
for y, m in [(2026, mm) for mm in range(4, 11)]:
    ms = dt.date(y, m, 1); me = (dt.date(y + (m == 12), m % 12 + 1, 1) - dt.timedelta(days=1)); dim = me.day
    ny, nm = (y, m + 1) if m < 12 else (y + 1, 1)
    pdate = first_friday(ny, nm).isoformat()
    for eid, name, role, dept, level, loc, mgr, hire, term, sal, pt, pl in DB.execute("SELECT * FROM employees").fetchall():
        a, b = max(d(hire), ms), min(d(term) if term else me, me)
        if b < a: continue
        frac = ((b - a).days + 1) / dim
        base = sal / 12 * frac
        pay_rows.append((f"{y}-{m:02d}", eid, round(base, 2), round(base * BEN_PCT[level], 2), round(base * TAX, 2), pdate))
DB.executemany("INSERT INTO payroll_runs VALUES (?,?,?,?,?,?)", pay_rows)
q("SELECT period_month, pay_date, COUNT(*) people, ROUND(SUM(base_pay+benefits_equity+employer_tax)) total FROM payroll_runs GROUP BY 1,2 ORDER BY 1")

,period_month,pay_date,people,total
0,2026-04,2026-05-01,42,610297.0
1,2026-05,2026-06-05,42,610297.0
2,2026-06,2026-07-03,42,602773.0
3,2026-07,2026-08-07,41,597756.0
4,2026-08,2026-09-04,42,601631.0
5,2026-09,2026-10-02,41,596112.0
6,2026-10,2026-11-06,42,600953.0


## 5. Data-quality checks
Ten checks, each returning the number of violating rows. They must all be 0 before any KPI is trusted. Below, a negative test injects three deliberate errors into a copy of the database to show the checks actually fire.

In [6]:
CHECKS = {
 "term_before_hire":        "SELECT COUNT(*) FROM employees WHERE term_date IS NOT NULL AND term_date < hire_date",
 "missing_hire_date":       "SELECT COUNT(*) FROM employees WHERE hire_date IS NULL OR hire_date = ''",
 "manager_not_found":       "SELECT COUNT(*) FROM employees WHERE manager_id IS NOT NULL AND manager_id NOT IN (SELECT employee_id FROM employees)",
 "manager_hired_after_report": "SELECT COUNT(*) FROM employees e JOIN employees m ON m.employee_id = e.manager_id WHERE m.hire_date > e.hire_date",
 "pipeline_dates_out_of_order": '''SELECT COUNT(*) FROM candidates WHERE
      (screen_date IS NOT NULL AND screen_date < applied_date) OR (interview_date IS NOT NULL AND interview_date < screen_date)
   OR (offer_date IS NOT NULL AND offer_date < interview_date) OR (accepted_date IS NOT NULL AND accepted_date < offer_date)
   OR (start_date IS NOT NULL AND start_date < accepted_date)''',
 "candidate_before_req_opened": "SELECT COUNT(*) FROM candidates c JOIN requisitions r USING(req_id) WHERE c.applied_date < r.opened_date",
 "candidate_future_dates":  "SELECT COUNT(*) FROM candidates WHERE applied_date > :ref OR screen_date > :ref OR interview_date > :ref OR offer_date > :ref OR accepted_date > :ref",
 "paid_when_not_employed":  '''SELECT COUNT(*) FROM payroll_runs p JOIN employees e USING(employee_id)
                               WHERE e.hire_date > date(p.period_month || '-01', '+1 month', '-1 day')
                                  OR (e.term_date IS NOT NULL AND e.term_date < p.period_month || '-01')''',
 "active_but_unpaid_latest": '''SELECT COUNT(*) FROM employees e WHERE e.term_date IS NULL AND e.hire_date <= :ref
                               AND NOT EXISTS (SELECT 1 FROM payroll_runs p WHERE p.employee_id = e.employee_id AND p.period_month = '2026-10')''',
 "leave_after_termination": "SELECT COUNT(*) FROM time_off t JOIN employees e USING(employee_id) WHERE e.term_date IS NOT NULL AND t.leave_date > e.term_date",
}
def run_checks(conn):
    out = []
    for name, sql in CHECKS.items():
        n = conn.execute(sql, {"ref": REF.isoformat()}).fetchall()[0][0]
        out.append((name, n, "PASS" if n == 0 else "FAIL"))
    return pd.DataFrame(out, columns=["check", "violations", "status"])

dq = run_checks(DB); display(dq)
assert (dq.violations == 0).all(), "fix the data before computing KPIs"

# negative test on a copy: three injected errors must be caught
bad = sqlite3.connect(":memory:"); bad.executescript(chr(10).join(DB.iterdump()))   # copy of the database
bad.execute("UPDATE employees SET term_date = '2020-01-01' WHERE employee_id = 5")                       # left before hire
bad.execute("UPDATE candidates SET screen_date = '2026-01-01' WHERE candidate_id = 9")                   # screen before applied
bad.execute("DELETE FROM payroll_runs WHERE employee_id = 7 AND period_month = '2026-10'")                # active employee missing from run
neg = run_checks(bad); print("Negative test (expect FAIL rows):"); display(neg[neg.status == "FAIL"])
assert (neg.status == "FAIL").sum() >= 3

,check,violations,status
0,term_before_hire,0,PASS
1,missing_hire_date,0,PASS
2,manager_not_found,0,PASS
3,manager_hired_after_report,0,PASS
4,pipeline_dates_out_of_order,0,PASS
5,candidate_before_req_opened,0,PASS
6,candidate_future_dates,0,PASS
7,paid_when_not_employed,0,PASS
8,active_but_unpaid_latest,0,PASS
9,leave_after_termination,0,PASS


Negative test (expect FAIL rows):


,check,violations,status
0,term_before_hire,1,FAIL
4,pipeline_dates_out_of_order,1,FAIL
7,paid_when_not_employed,7,FAIL
8,active_but_unpaid_latest,1,FAIL
9,leave_after_termination,5,FAIL


## 6. KPI definitions and SQL
Windows are **trailing** from the reference date: week = 7 days, month = 30 days, quarter = 90 days. Definitions are documented in `METHODOLOGY.md`.

In [7]:
REFS = REF.isoformat(); WIN = {"week": 7, "month": 30, "quarter": 90}
def ago(n): return (REF - dt.timedelta(days=n)).isoformat()
def one(sql, **p): return DB.execute(sql, p).fetchall()[0][0]

# ---- Headcount (active at REF) and joiners per window
HEADCOUNT = one("SELECT COUNT(*) FROM employees WHERE hire_date <= :ref AND (term_date IS NULL OR term_date > :ref)", ref=REFS)
JOINED = {w: one("SELECT COUNT(*) FROM employees WHERE hire_date > :a AND hire_date <= :ref", a=ago(n), ref=REFS) for w, n in WIN.items()}

# ---- Open roles, final-round interviews, screens held (7d), roles opened (90d)
OPEN_ROLES = one("SELECT COUNT(*) FROM requisitions WHERE status = 'open' AND opened_date <= :ref", ref=REFS)
FINAL_ROUND = one("SELECT COUNT(*) FROM candidates WHERE stage = 'Interview' AND final_round = 1")
SCREENS_7D = one("SELECT COUNT(*) FROM candidates WHERE screen_date > :a AND screen_date <= :ref", a=ago(7), ref=REFS)
OPENED_90D = one("SELECT COUNT(*) FROM requisitions WHERE opened_date > :a AND opened_date <= :ref", a=ago(90), ref=REFS)

# ---- 12-month retention (cohort method) now and one year earlier
RET_SQL = '''
WITH cohort AS (SELECT employee_id, term_date FROM employees
                WHERE hire_date <= :asof AND (term_date IS NULL OR term_date > :asof))
SELECT COUNT(*) AS cohort_n,
       SUM(CASE WHEN term_date IS NULL OR term_date > :end THEN 1 ELSE 0 END) AS retained_n FROM cohort'''
def retention(asof, end):
    n, k = DB.execute(RET_SQL, {"asof": asof, "end": end}).fetchall()[0]; return n, k, 100.0 * k / n
RET_NOW, RET_PRIOR = retention(ago(365), REFS), retention(ago(730), ago(365))
RET_DELTA = RET_NOW[2] - RET_PRIOR[2]
DEPART = {w: one("SELECT COUNT(*) FROM employees WHERE term_date > :a AND term_date <= :ref", a=ago(n), ref=REFS) for w, n in WIN.items()}

# ---- Payroll: latest period, previous month, Q2 average run-rate
PAY_SQL = "SELECT SUM(base_pay + benefits_equity + employer_tax) FROM payroll_runs WHERE period_month = :m"
PAY_OCT, PAY_SEP = one(PAY_SQL, m="2026-10"), one(PAY_SQL, m="2026-09")
PAY_Q2 = sum(one(PAY_SQL, m=m) for m in ["2026-04","2026-05","2026-06"]) / 3
NEXT_RUN = one("SELECT MAX(pay_date) FROM payroll_runs")

# ---- Median time-to-offer (days, application to offer) for offers extended in the last 365 days; window functions
TTO_SQL = '''
WITH t AS (SELECT julianday(offer_date) - julianday(applied_date) AS days FROM candidates
           WHERE offer_date IS NOT NULL AND offer_date > :a AND offer_date <= :ref),
r AS (SELECT days, ROW_NUMBER() OVER (ORDER BY days) AS rn, COUNT(*) OVER () AS n FROM t)
SELECT AVG(days), MAX(n) FROM r WHERE rn IN ((n + 1) / 2, (n + 2) / 2)'''
TTO, TTO_N = DB.execute(TTO_SQL, {"a": ago(365), "ref": REFS}).fetchall()[0]

# ---- Pipeline snapshot
PIPE = dict(DB.execute('''SELECT CASE WHEN stage = 'Hired' THEN 'Hired' ELSE stage END, COUNT(*) FROM candidates
        WHERE stage IN ('Applied','Screen','Interview','Offer') OR (stage = 'Hired' AND start_date > :ref) GROUP BY 1''', {"ref": REFS}).fetchall())
ACTIVE_CANDS = sum(PIPE.get(s, 0) for s in ["Applied","Screen","Interview","Offer"])
STARTS_NOV = one("SELECT COUNT(*) FROM candidates WHERE stage = 'Hired' AND start_date BETWEEN '2026-11-01' AND '2026-11-30'")

summary = pd.DataFrame([
  ("Headcount", HEADCOUNT), ("Joined (7d / 30d / 90d)", f"{JOINED['week']} / {JOINED['month']} / {JOINED['quarter']}"),
  ("Open roles", OPEN_ROLES), ("Final-round interviews", FINAL_ROUND), ("Screens held (7d)", SCREENS_7D), ("Roles opened (90d)", OPENED_90D),
  ("Retention now (kept / cohort)", f"{RET_NOW[2]:.1f}%  ({RET_NOW[1]}/{RET_NOW[0]})"), ("Retention a year earlier", f"{RET_PRIOR[2]:.1f}%  ({RET_PRIOR[1]}/{RET_PRIOR[0]})"),
  ("Departures (7d / 30d / 90d)", f"{DEPART['week']} / {DEPART['month']} / {DEPART['quarter']}"),
  ("Payroll Oct / Sep / Q2 avg ($)", f"{PAY_OCT:,.0f} / {PAY_SEP:,.0f} / {PAY_Q2:,.0f}"), ("Next run", NEXT_RUN),
  ("Median time-to-offer (days, n)", f"{TTO:.1f}  (n={TTO_N})"), ("Active candidates / starts in Nov", f"{ACTIVE_CANDS} / {STARTS_NOV}"), ("Pipeline by stage", PIPE)],
  columns=["KPI", "Value"]); summary

,KPI,Value
0,Headcount,42
1,Joined (7d / 30d / 90d),0 / 1 / 2
2,Open roles,9
3,Final-round interviews,4
4,Screens held (7d),5
5,Roles opened (90d),10
6,Retention now (kept / cohort),93.0% (40/43)
7,Retention a year earlier,91.9% (34/37)
8,Departures (7d / 30d / 90d),0 / 0 / 1
9,Payroll Oct / Sep / Q2 avg ($),"600,953 / 596,112 / 607,789"


### Window-function examples: month-end headcount with LAG, and payroll share by department

In [8]:
TREND = q('''
WITH RECURSIVE months(m_start) AS (SELECT '2025-09-01' UNION ALL SELECT date(m_start, '+1 month') FROM months WHERE m_start < '2026-10-01'),
hc AS (SELECT substr(m.m_start, 1, 7) AS month, COUNT(e.employee_id) AS headcount
       FROM months m LEFT JOIN employees e ON e.hire_date <= date(m.m_start, '+1 month', '-1 day')
            AND (e.term_date IS NULL OR e.term_date > date(m.m_start, '+1 month', '-1 day'))
       GROUP BY m.m_start)
SELECT month, headcount, headcount - LAG(headcount) OVER (ORDER BY month) AS net_change FROM hc ORDER BY month''')
display(TREND)

PAY_DEPT = q('''
SELECT d.dept_id, d.name, COUNT(*) AS people,
       ROUND(SUM(p.base_pay + p.benefits_equity + p.employer_tax)) AS monthly_cost,
       ROUND(100.0 * SUM(p.base_pay + p.benefits_equity + p.employer_tax) / SUM(SUM(p.base_pay + p.benefits_equity + p.employer_tax)) OVER (), 1) AS pct_of_total
FROM payroll_runs p JOIN employees e USING(employee_id) JOIN departments d ON d.dept_id = e.dept_id
WHERE p.period_month = '2026-10' GROUP BY d.dept_id ORDER BY monthly_cost DESC''')
display(PAY_DEPT)
SPLIT = q("SELECT SUM(base_pay) b, SUM(benefits_equity) be, SUM(employer_tax) t FROM payroll_runs WHERE period_month = '2026-10'").iloc[0]
SPLIT_PCT = [100 * SPLIT.b / PAY_OCT, 100 * SPLIT.be / PAY_OCT, 100 * SPLIT.t / PAY_OCT]
print("split % base / benefits+equity / employer tax:", [round(x, 1) for x in SPLIT_PCT])

,month,headcount,net_change
0,2025-09,43,NaN
1,2025-10,43,0.0
2,2025-11,43,0.0
3,2025-12,43,0.0
4,2026-01,43,0.0
5,2026-02,43,0.0
6,2026-03,42,-1.0
7,2026-04,42,0.0
8,2026-05,42,0.0
9,2026-06,41,-1.0


,dept_id,name,people,monthly_cost,pct_of_total
0,eng,Engineering,13,187465.0,31.2
1,prod,Product,6,91230.0,15.2
2,des,Design,6,87975.0,14.6
3,sales,Sales,6,87809.0,14.6
4,ops,People & Ops,7,85270.0,14.2
5,fin,Finance,4,61204.0,10.2


split % base / benefits+equity / employer tax: [np.float64(77.2), np.float64(17.4), np.float64(5.4)]


## 7. Three insights
Each one is a question an HR or finance partner could ask, answered from the synthetic data. They show method, not real-world findings.

In [9]:
# Insight 1: which hiring step is slowest? median days per step for offers extended in the last 365 days (window functions)
I1 = q('''
WITH g AS (SELECT julianday(screen_date) - julianday(applied_date) AS s1,
                  julianday(interview_date) - julianday(screen_date) AS s2,
                  julianday(offer_date) - julianday(interview_date) AS s3
           FROM candidates WHERE offer_date IS NOT NULL AND offer_date > :a AND offer_date <= :ref),
u AS (SELECT 'Application to screen' AS step, s1 AS days FROM g UNION ALL SELECT 'Screen to interview', s2 FROM g UNION ALL SELECT 'Interview to offer', s3 FROM g),
r AS (SELECT step, days, ROW_NUMBER() OVER (PARTITION BY step ORDER BY days) rn, COUNT(*) OVER (PARTITION BY step) n FROM u)
SELECT step, MAX(n) AS offers, ROUND(AVG(days), 1) AS median_days FROM r WHERE rn IN ((n + 1) / 2, (n + 2) / 2) GROUP BY step ORDER BY median_days DESC''', a=ago(365), ref=REFS)
display(I1)
I1_DEPT = q('''
WITH t AS (SELECT r.dept_id, julianday(c.offer_date) - julianday(c.applied_date) AS days
           FROM candidates c JOIN requisitions r USING(req_id) WHERE c.offer_date IS NOT NULL AND c.offer_date > :a AND c.offer_date <= :ref),
r AS (SELECT dept_id, days, ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY days) rn, COUNT(*) OVER (PARTITION BY dept_id) n FROM t)
SELECT dept_id, MAX(n) AS offers, ROUND(AVG(days), 1) AS median_days_to_offer FROM r WHERE rn IN ((n + 1) / 2, (n + 2) / 2) GROUP BY dept_id ORDER BY median_days_to_offer DESC''', a=ago(365), ref=REFS)
display(I1_DEPT)

# Insight 2: coverage risk. Share of each department out on the same upcoming day (leave calendar after the reference date)
I2 = q('''
WITH out AS (SELECT t.leave_date, e.dept_id, COUNT(*) AS people_out FROM time_off t JOIN employees e USING(employee_id)
             WHERE e.term_date IS NULL AND t.leave_date > :ref GROUP BY 1, 2),
hc AS (SELECT dept_id, COUNT(*) AS hc FROM employees WHERE term_date IS NULL GROUP BY 1)
SELECT o.leave_date, o.dept_id, o.people_out, hc.hc, ROUND(100.0 * o.people_out / hc.hc, 1) AS pct_out
FROM out o JOIN hc USING(dept_id) ORDER BY pct_out DESC, o.leave_date LIMIT 8''', ref=REFS)
display(I2)

# Insight 3: requisition aging. Open roles ranked by days open, with how far their best candidate has progressed
I3 = q('''
SELECT r.req_id, r.title, r.dept_id, CAST(julianday(:ref) - julianday(r.opened_date) AS INT) AS days_open,
       SUM(CASE WHEN c.stage IN ('Applied','Screen','Interview','Offer') THEN 1 ELSE 0 END) AS active_candidates,
       MAX(CASE c.stage WHEN 'Offer' THEN 4 WHEN 'Interview' THEN 3 WHEN 'Screen' THEN 2 WHEN 'Applied' THEN 1 ELSE 0 END) AS furthest_stage_rank
FROM requisitions r LEFT JOIN candidates c USING(req_id) WHERE r.status = 'open'
GROUP BY r.req_id ORDER BY days_open DESC''', ref=REFS)
display(I3)

,step,offers,median_days
0,Interview to offer,11,18.0
1,Screen to interview,11,11.0
2,Application to screen,11,6.0


,dept_id,offers,median_days_to_offer
0,prod,1,49.0
1,des,1,41.0
2,eng,4,35.0
3,ops,2,32.5
4,sales,2,31.5
5,fin,1,23.0


,leave_date,dept_id,people_out,hc,pct_out
0,2026-11-25,fin,1,4,25.0
1,2026-11-26,fin,1,4,25.0
2,2026-11-27,fin,1,4,25.0
3,2026-11-02,des,1,6,16.7
4,2026-11-02,prod,1,6,16.7
5,2026-11-03,prod,1,6,16.7
6,2026-11-04,prod,1,6,16.7
7,2026-11-05,sales,1,6,16.7


,req_id,title,dept_id,days_open,active_candidates,furthest_stage_rank
0,R106,Engineering Manager,eng,96,4,4
1,R102,Backend Engineer,eng,68,4,2
2,R105,Account Executive,sales,61,4,3
3,R101,Senior Data Analyst,prod,53,5,3
4,R103,Backend Engineer,eng,47,4,3
5,R104,Product Designer,des,40,4,3
6,R107,Financial Analyst,fin,33,4,4
7,R108,Customer Success Manager,sales,26,4,4
8,R109,Product Manager,prod,19,3,3


## 8. Dashboard feeds: out-this-week list (gaps-and-islands with business-day index)

In [10]:
OUT_START, OUT_END = "2026-11-02", "2026-11-08"
OUTWK = q('''
WITH l AS (SELECT t.employee_id, t.leave_type, t.leave_date, c.bd_index,
                  ROW_NUMBER() OVER (PARTITION BY t.employee_id, t.leave_type ORDER BY t.leave_date) AS rn
           FROM time_off t JOIN calendar c ON c.day = t.leave_date),
isl AS (SELECT employee_id, leave_type, MIN(leave_date) AS start_d, MAX(leave_date) AS end_d
        FROM (SELECT *, bd_index - rn AS grp FROM l) GROUP BY employee_id, leave_type, grp)
SELECT e.name, e.role, i.leave_type, i.start_d, i.end_d FROM isl i JOIN employees e USING(employee_id)
WHERE i.start_d <= :we AND i.end_d >= :ws ORDER BY i.start_d, e.name''', ws=OUT_START, we=OUT_END)
OUTWK

,name,role,leave_type,start_d,end_d
0,Felix Wagner,Design Engineer,per,2026-11-02,2026-11-02
1,Ingrid Larsen,Product Manager,vac,2026-11-02,2026-11-04
2,Marcus Webb,Software Engineer,sick,2026-11-03,2026-11-03
3,Sofia Petrova,Senior Software Engineer,par,2026-11-04,2026-11-10
4,Rob Callahan,Account Executive,vac,2026-11-05,2026-11-06
5,Yuki Mori,Product Designer,vac,2026-11-06,2026-11-06


## 9. Cross-check with an independent pandas calculation
Headline numbers are recomputed without SQL; any mismatch stops the notebook.

In [11]:
E = pd.read_sql_query("SELECT * FROM employees", DB, parse_dates=["hire_date", "term_date"])
P = pd.read_sql_query("SELECT * FROM payroll_runs", DB)
ref = pd.Timestamp(REF)
active = E[(E.hire_date <= ref) & (E.term_date.isna() | (E.term_date > ref))]
assert len(active) == HEADCOUNT
asof = ref - pd.Timedelta(days=365)
coh = E[(E.hire_date <= asof) & (E.term_date.isna() | (E.term_date > asof))]
kept = coh[coh.term_date.isna() | (coh.term_date > ref)]
assert (len(coh), len(kept)) == (RET_NOW[0], RET_NOW[1])
oct_total = P[P.period_month == "2026-10"][["base_pay", "benefits_equity", "employer_tax"]].sum().sum()
assert abs(oct_total - PAY_OCT) < 0.01
C = pd.read_sql_query("SELECT applied_date, offer_date FROM candidates WHERE offer_date IS NOT NULL", DB, parse_dates=["applied_date", "offer_date"])
C = C[(C.offer_date > asof) & (C.offer_date <= ref)]
assert abs((C.offer_date - C.applied_date).dt.days.median() - TTO) < 1e-9
print("pandas cross-check passed: headcount, retention cohort, October payroll, median time-to-offer")

pandas cross-check passed: headcount, retention cohort, October payroll, median time-to-offer


## 10. Export tables and KPI file

In [12]:
for t in ["departments", "employees", "time_off", "requisitions", "candidates", "payroll_runs"]:
    pd.read_sql_query(f"SELECT * FROM {t}", DB).to_csv(DATA / f"{t}.csv", index=False)

# department $K rounded by largest remainder so the rows add up exactly to the rounded total shown on the page
_raw = {r.dept_id: r.monthly_cost / 1000 for r in PAY_DEPT.itertuples()}
pay_k = {k: int(v) for k, v in _raw.items()}
for k in sorted(_raw, key=lambda k: _raw[k] - pay_k[k], reverse=True)[: round(PAY_OCT / 1000) - sum(pay_k.values())]:
    pay_k[k] += 1
KPI = dict(reference_date=REFS, headcount=HEADCOUNT, joined=JOINED, open_roles=OPEN_ROLES, final_round=FINAL_ROUND, screens_7d=SCREENS_7D,
           opened_90d=OPENED_90D, retention_pct=round(RET_NOW[2], 2), retention_prior_pct=round(RET_PRIOR[2], 2), retention_delta_pts=round(RET_DELTA, 2),
           retention_cohort=[RET_NOW[0], RET_NOW[1]], departures=DEPART, payroll_oct=round(PAY_OCT, 2), payroll_sep=round(PAY_SEP, 2),
           payroll_q2_avg=round(PAY_Q2, 2), next_run=NEXT_RUN, median_time_to_offer_days=round(TTO, 1), offers_in_window=int(TTO_N),
           pipeline=PIPE, active_candidates=ACTIVE_CANDS, starts_in_november=STARTS_NOV, payroll_dept_k=pay_k, split_pct=[round(x, 1) for x in SPLIT_PCT])
(DATA / "kpis.json").write_text(json.dumps(KPI, indent=2))
print(json.dumps(KPI, indent=1)[:900])

{
 "reference_date": "2026-10-31",
 "headcount": 42,
 "joined": {
  "week": 0,
  "month": 1,
  "quarter": 2
 },
 "open_roles": 9,
 "final_round": 4,
 "screens_7d": 5,
 "opened_90d": 10,
 "retention_pct": 93.02,
 "retention_prior_pct": 91.89,
 "retention_delta_pts": 1.13,
 "retention_cohort": [
  43,
  40
 ],
 "departures": {
  "week": 0,
  "month": 0,
  "quarter": 1
 },
 "payroll_oct": 600953.19,
 "payroll_sep": 596112.06,
 "payroll_q2_avg": 607789.22,
 "next_run": "2026-11-06",
 "median_time_to_offer_days": 34.0,
 "offers_in_window": 11,
 "pipeline": {
  "Applied": 18,
  "Hired": 2,
  "Interview": 6,
  "Offer": 3,
  "Screen": 9
 },
 "active_candidates": 36,
 "starts_in_november": 2,
 "payroll_dept_k": {
  "eng": 188,
  "prod": 91,
  "des": 88,
  "sales": 88,
  "ops": 85,
  "fin": 61
 },
 "split_pct": [
  77.2,
  17.4,
  5.4
 ]
}


## 11. Build `index.html` from the template
Every hard-coded KPI string in the Muse template is replaced with the value computed above. Each replacement asserts that the target text exists exactly the expected number of times, so a template change cannot silently skip a number.

In [13]:
out = TEMPLATE.read_text(encoding="utf8")
def sub(old, new, count=1):
    global out
    assert out.count(old) == count, f"expected {count}x, found {out.count(old)}x: {old[:70]!r}"
    out = out.replace(old, new)
def rsub(pattern, new, flags=re.S):
    global out
    out, n = re.subn(pattern, lambda m: new, out, count=1, flags=flags); assert n == 1, pattern

def fmt_d(x): x = d(x) if isinstance(x, str) else x; return f"{x.strftime('%b')} {x.day}"
def arrow(v): return "▲" if v >= 0 else "▼"
def joined_txt(n, w): return f"▲ {n} joined this {w}" if n else f"No new joiners this {w}"

# KPI cards
sub('<span data-count="42">0</span>', f'<span data-count="{HEADCOUNT}">0</span>')
sub('<span data-count="9">0</span>', f'<span data-count="{OPEN_ROLES}">0</span>')
sub('<span data-count="94">0</span>', f'<span data-count="{round(RET_NOW[2])}">0</span>')
sub('$<span data-count="612">0</span>', f'$<span data-count="{round(PAY_OCT/1000)}">0</span>')
sub("▲ 3 joined this month", joined_txt(JOINED["month"], "month")); sub("▲ 1 joined this week", joined_txt(JOINED["week"], "week"))
sub("▲ 6 joined this quarter", joined_txt(JOINED["quarter"], "quarter"))
sub("4 in final interview rounds", f"{FINAL_ROUND} in final interview rounds"); sub("2 screens booked this week", f"{SCREENS_7D} screens held this week")
sub("11 roles opened this quarter", f"{OPENED_90D} roles opened this quarter")
sub("▲ 1.2 pts vs last year", f"{arrow(RET_DELTA)} {abs(RET_DELTA):.1f} pts vs last year")
sub("0 departures this week", f"{DEPART['week']} departure{'s' if DEPART['week'] != 1 else ''} this week")
sub("▲ 0.8 pts this quarter", f"{DEPART['quarter']} departure{'s' if DEPART['quarter'] != 1 else ''} this quarter")
pct = lambda a, b: 100 * (a - b) / b
sub("▲ 4.1% after Oct hires", f"{arrow(pct(PAY_OCT, PAY_SEP))} {abs(pct(PAY_OCT, PAY_SEP)):.1f}% vs Sep run")
sub("Next run: Fri, Nov 6", f"Next run: {d(NEXT_RUN).strftime('%a')}, {fmt_d(NEXT_RUN)}")
sub("▲ 9.6% vs Q2 run-rate", f"{arrow(pct(PAY_OCT, PAY_Q2))} {abs(pct(PAY_OCT, PAY_Q2)):.1f}% vs Q2 run-rate")

# Directory data: regenerate RAW from the employees table (tenure now derived from hire_date)
act = q("SELECT * FROM employees WHERE term_date IS NULL ORDER BY employee_id")
raw_lines = []
for r in act.itertuples():
    tenure = round((REF - d(r.hire_date)).days / 365.25, 1)
    mgr = int(r.manager_id) - 1 if pd.notna(r.manager_id) else -1
    raw_lines.append("        " + json.dumps([r.name, r.role, r.dept_id, int(r.level), r.location, tenure, mgr, int(r.pto_left_days), int(r.pto_total_days)], ensure_ascii=False))
rsub(r"(var RAW = \[\n).*?(\n      \];)", "var RAW = [\n" + ",\n".join(raw_lines) + "\n      ];")

# Out this week
def rng_txt(s, e):
    s, e = d(s), d(e)
    if s == e: return f"{s.strftime('%a')}, {fmt_d(s)}"
    return f"{fmt_d(s)}–{e.day}" if s.month == e.month else f"{fmt_d(s)} – {fmt_d(e)}"
items = []
for r in OUTWK.itertuples():
    items.append('        { n:%s, r:%s, t:%s }' % (json.dumps(r.name, ensure_ascii=False), json.dumps(r.role.replace("Senior", "Sr.") + " · " + rng_txt(r.start_d, r.end_d), ensure_ascii=False), json.dumps(r.leave_type)))
rsub(r"var OUT_WEEK = \[.*?\n      \];", "var OUT_WEEK = [\n" + ",\n".join(items) + "\n      ];")
sub("Nov 2 – Nov 8 · 6 people away", f"Nov 2 – Nov 8 · {OUTWK.name.nunique()} people away")

# Pipeline
sub("38 active candidates across 9 open roles. Three offers out, two starts confirmed for November.",
    f"{ACTIVE_CANDS} active candidates across {OPEN_ROLES} open roles. {PIPE['Offer']} offers out, {STARTS_NOV} starts confirmed for November.")
sub("Median time-to-offer: 24 days", f"Median time-to-offer: {TTO:.0f} days")
for s in ["Applied", "Screen", "Interview", "Offer", "Hired"]:
    rsub(r'(name:"%s",\s+count:)\d+(,)' % s, "name:\"%s\", count:%d," % (s, PIPE[s]))
sub("s.count/18*100", f"s.count/{max(PIPE.values())}*100")
sub('{ n:"Jonah Fields", r:"Financial Analyst", tags:[["Verbal yes","good"]] }',
    '{ n:"Jonah Fields", r:"Financial Analyst", tags:[["Verbal yes","good"]] },\n          { n:"Priti Menon", r:"Customer Success Manager", tags:[["Offer out",""]] }')

# Payroll
order = ["eng", "sales", "prod", "des", "ops", "fin"]
order = sorted(pay_k, key=lambda k: -pay_k[k])
rsub(r"var PAYROLL = \[.*?\];", "var PAYROLL = [\n" + ",\n".join('        { dept:"%s", monthly:%d }' % (k, pay_k[k]) for k in order) + "\n      ];")
tot_k = round(PAY_OCT / 1000); avg_m = PAY_OCT / HEADCOUNT
assert sum(pay_k.values()) == tot_k, "department rows must sum to the total shown"
sub("Gross pay for the fictional October 2026 run, before the November hires land.",
    "Employer cost (base pay, benefits &amp; equity, employer taxes) for the October 2026 period, paid " + fmt_d(NEXT_RUN) + ". Synthetic data.")
sub('<div class="t-value" id="pay-total-value">$612K</div>', f'<div class="t-value" id="pay-total-value">${tot_k}K</div>')
sub("Across 42 people · avg $14.6K / person / month", f"Across {HEADCOUNT} people · avg ${avg_m/1000:.1f}K / person / month", count=2)
sub("Across 42 people · avg $175K / person / year", f"Across {HEADCOUNT} people · avg ${avg_m*12/1000:.0f}K / person / year")
sub("width:78%;background:var(--teal)", f"width:{SPLIT_PCT[0]:.0f}%;background:var(--teal)"); sub("width:15%;background:var(--amber)", f"width:{SPLIT_PCT[1]:.0f}%;background:var(--amber)")
sub("width:7%;background:var(--coral)", f"width:{SPLIT_PCT[2]:.0f}%;background:var(--coral)")
sub('Base salary<span class="pct">78%</span>', f'Base salary<span class="pct">{SPLIT_PCT[0]:.0f}%</span>')
sub('Benefits &amp; equity<span class="pct">15%</span>', f'Benefits &amp; equity<span class="pct">{SPLIT_PCT[1]:.0f}%</span>')
sub('Employer taxes<span class="pct">7%</span>', f'Employer taxes<span class="pct">{SPLIT_PCT[2]:.0f}%</span>')
print("replacements applied; pipeline", PIPE, "| payroll $K", pay_k)

replacements applied; pipeline {'Applied': 18, 'Hired': 2, 'Interview': 6, 'Offer': 3, 'Screen': 9} | payroll $K {'eng': 188, 'prod': 91, 'des': 88, 'sales': 88, 'ops': 85, 'fin': 61}


In [14]:
# Insights panel (static HTML, text built from the query results above)
longest = I1.iloc[0]; shortest = I1.iloc[-1]
dep_lo, dep_hi = I1_DEPT.median_days_to_offer.min(), I1_DEPT.median_days_to_offer.max()
i1 = (f"Across {int(longest.offers)} offers in the last year the slowest step is {longest.step.lower()} (median {longest.median_days:.0f} days), "
      f"against {shortest.median_days:.0f} days for {shortest.step.lower()}. Department medians range {dep_lo:.0f} to {dep_hi:.0f} days but rest on 1 to 4 offers each, so treat them as directional.")
top = I2.iloc[0]; same = I2[(I2.dept_id == top.dept_id) & (I2.pct_out == top.pct_out)]
i2 = (f"{DEPT_NAMES[top.dept_id]} has {int(top.hc)} people, so a single absence is {top.pct_out:.0f}% of the team. That happens on {len(same)} upcoming day(s), "
      f"starting {fmt_d(same.leave_date.min())}. Small teams need named backups for approvals and month-end work.")
oldest = I3.iloc[0]; stuck = I3[(I3.active_candidates > 0) & (I3.furthest_stage_rank <= 2)]
stuck_txt = "; ".join(f"{r.title} ({int(r.days_open)} days)" for r in stuck.itertuples()) or "none"
i3 = (f"{oldest.title} ({DEPT_NAMES[oldest.dept_id]}) has been open {int(oldest.days_open)} days, the longest of {len(I3)} open roles, and already has an offer-stage candidate. "
      f"Roles with candidates but none past the screen stage: {stuck_txt}. Those are where sourcing or screening effort would help first.")
cards = "".join(f'<div class="insight"><div class="i-tag">{t}</div><h3>{h}</h3><p>{p}</p></div>' for t, h, p in [
   ("Hiring speed", "Where offers take longest", i1), ("Coverage risk", "When a team is thinnest", i2), ("Requisition aging", "Which roles need attention", i3)])
panel = ('<section class="panel" style="margin-top:22px" aria-label="Insights"><div class="panel-head"><div><h2>Insights</h2>'
         '<div class="sub">Three questions answered with SQL on synthetic data. Method demonstration, not real results.</div></div></div>'
         f'<div class="insights">{cards}</div></section>\n\n      ')
sub('<footer class="foot">', panel + '<footer class="foot">')
css = '''
      .insights{padding:6px 22px 22px;display:grid;grid-template-columns:repeat(3,1fr);gap:14px}
      .insight{background:var(--panel-2);border:1px solid var(--line);border-radius:14px;padding:16px 18px}
      .insight .i-tag{font-size:11px;font-weight:800;letter-spacing:.07em;text-transform:uppercase;color:var(--muted)}
      .insight h3{font-size:15px;font-weight:700;margin:6px 0 8px}
      .insight p{font-size:13.5px;line-height:1.55;color:var(--ink-2,var(--muted));font-weight:500}
      @media (max-width:860px){.insights{grid-template-columns:1fr}}
'''
i = out.index("</style>"); out = out[:i] + css + "    " + out[i:]
sub("Every person, candidate and figure on this page is fictional demo data.",
    "Every person, candidate and figure on this page is fictional demo data; KPIs are computed in SQL from a synthetic dataset (see the repository notebook).")
pathlib.Path("index.html").write_text(out, encoding="utf8")
print("index.html written:", len(out)//1024, "KB")

index.html written: 1192 KB
